# 3D U-Net Baseline — MAMA-MIA Breast MRI Segmentation

**Model:** 3D U-Net (22.5M params)  
**Dataset:** MAMA-MIA (1200 train / 306 test)  
**Goal:** Beat paper baseline Dice of 0.762  
**GPU:** Kaggle P100 (16 GB VRAM)

Each Kaggle session covers ~100 epochs (~9h). Use `RESUME_FROM` to continue across sessions.

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURATION — edit these before running
# ─────────────────────────────────────────────

DATA_ROOT    = "/kaggle/input/datasets/bharathkumarvemuri/mama-mia-breast-mri-segmentation/mama_mia_kaggle_upload"
OUTPUT_DIR   = "/kaggle/working/outputs/unet3d"      # checkpoints + logs saved here
CACHE_DIR    = "/kaggle/working/cache"               # preprocessed volume cache (reused across epochs)
CODE_DIR     = "/kaggle/working/FADC-3D"             # cloned repo

EPOCHS       = 300
BATCH_SIZE   = 2
NUM_WORKERS  = 4

# To resume training from a previous session:
# 1. Add your previous session's output as a Kaggle input dataset
# 2. Set RESUME_FROM to the checkpoint path, e.g.:
#    RESUME_FROM = "/kaggle/input/unet3d-checkpoint/latest_checkpoint.pth"
RESUME_FROM  = None

In [ ]:
# ─────────────────────────────────────────────
# 1. INSTALL DEPENDENCIES
# ─────────────────────────────────────────────
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "monai", "-q"], check=True)
print("Dependencies ready.")

In [ ]:
# ─────────────────────────────────────────────
# 2. CLONE CODE FROM GITHUB
# ─────────────────────────────────────────────
import os

if os.path.exists(CODE_DIR):
    print("Repo already exists — pulling latest...")
    os.system(f"git -C {CODE_DIR} pull")
else:
    os.system(f"git clone https://github.com/BHARATH-VEMURI/FADC-3D.git {CODE_DIR}")
    print("Repo cloned.")

sys.path.insert(0, CODE_DIR)
print(f"Code path: {CODE_DIR}")

In [ ]:
# ─────────────────────────────────────────────
# 3. VERIFY GPU
# ─────────────────────────────────────────────
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU             : {gpu.name}")
    print(f"VRAM            : {gpu.total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found — enable GPU in Kaggle settings (Settings → Accelerator → GPU P100)")

In [ ]:
# ─────────────────────────────────────────────
# 4. VERIFY DATASET
# ─────────────────────────────────────────────
from pathlib import Path

data_path  = Path(DATA_ROOT)
images_dir = data_path / "images"
seg_dir    = data_path / "segmentations" / "expert"
csv_path   = data_path / "train_test_splits.csv"

patient_folders = [f for f in images_dir.iterdir() if f.is_dir()]
seg_files       = list(seg_dir.glob("*.nii.gz")) + list(seg_dir.glob("*.nii"))

print(f"Patient folders : {len(patient_folders)}")
print(f"Segmentations   : {len(seg_files)}")
print(f"Split CSV exists: {csv_path.exists()}")

# Sample one image to confirm extension
sample = sorted(patient_folders)[0]
sample_files = list(sample.iterdir())
print(f"Sample patient  : {sample.name} → {[f.name for f in sample_files]}")

collections = {}
for f in patient_folders:
    col = f.name.split("_")[0]
    collections[col] = collections.get(col, 0) + 1
for col, count in sorted(collections.items()):
    print(f"  {col}: {count} patients")

In [ ]:
# ─────────────────────────────────────────────
# 5. RUN TRAINING
# Preprocessing cache is built on the FIRST run (takes ~40 min for 1200 cases).
# All subsequent epochs load from cache — much faster.
# ─────────────────────────────────────────────
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR,  exist_ok=True)

train_script = os.path.join(CODE_DIR, "training", "train_centralized.py")

cmd = [
    sys.executable, train_script,
    "--data_root",           DATA_ROOT,
    "--output_dir",          OUTPUT_DIR,
    "--persistent_cache_dir", CACHE_DIR,
    "--epochs",              str(EPOCHS),
    "--batch_size",          str(BATCH_SIZE),
    "--num_workers",         str(NUM_WORKERS),
]

if RESUME_FROM:
    cmd += ["--resume", RESUME_FROM]

print("Command:", " ".join(cmd))
print("=" * 60)

result = subprocess.run(cmd, check=False)
print(f"\nExit code: {result.returncode}")

In [ ]:
# ─────────────────────────────────────────────
# 6. PLOT TRAINING CURVES
# ─────────────────────────────────────────────
import json
import matplotlib.pyplot as plt

log_path = os.path.join(OUTPUT_DIR, "train_log.json")

if not os.path.exists(log_path):
    print("No training log found yet.")
else:
    with open(log_path) as f:
        log = json.load(f)

    epochs    = [e["epoch"]    for e in log]
    losses    = [e["loss"]     for e in log]
    val_epochs = [e["epoch"]   for e in log if "val_dice" in e]
    val_dices  = [e["val_dice"] for e in log if "val_dice" in e]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, losses, color="steelblue", linewidth=1.5)
    ax1.set_title("Training Loss", fontsize=13)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.grid(True, alpha=0.3)

    ax2.plot(val_epochs, val_dices, color="darkorange", linewidth=1.5, marker="o", markersize=4)
    ax2.axhline(y=0.762, color="red", linestyle="--", linewidth=1, label="Paper baseline (0.762)")
    ax2.set_title("Validation Dice", fontsize=13)
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Dice Score")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    if val_dices:
        best = max(val_dices)
        ax2.set_title(f"Validation Dice  (best: {best:.4f})", fontsize=13)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
    plt.show()
    print(f"Epochs completed: {len(log)}")
    if val_dices:
        print(f"Best Val Dice   : {max(val_dices):.4f}  (paper baseline: 0.762)")